In [1]:
from sklearn.base import clone
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

def make_data():
    d = load_wine()
    return d.data, d.target

def scaled_vs_unscaled(estimator, X, y, seed=0):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.30, random_state=seed, stratify=y
    )
    # unscaled
    model_unscaled = clone(estimator)
    model_unscaled.fit(X_train, y_train)
    preds_unscaled = model_unscaled.predict(X_test)
    acc_unscaled = round(float(accuracy_score(y_test, preds_unscaled)), 4)
    
    # scaled
    model_scaled = make_pipeline(StandardScaler(), clone(estimator))
    model_scaled.fit(X_train, y_train)
    preds_scaled = model_scaled.predict(X_test)
    acc_scaled = round(float(accuracy_score(y_test, preds_scaled)), 4)
    
    return {"unscaled": acc_unscaled, "scaled": acc_scaled}

def needs_scaling(estimator, X, y, seed=0, threshold=0.02):
    res = scaled_vs_unscaled(estimator, X, y, seed=seed)
    return bool(round(res["scaled"] - res["unscaled"], 4) > threshold)

def ablation(X, y, seed=0):
    models = {
        "knn": KNeighborsClassifier(5),
        "svm": SVC(),
        "tree": DecisionTreeClassifier(random_state=0),
        "forest": RandomForestClassifier(n_estimators=100, random_state=0)
    }
    table = {}
    for name, est in models.items():
        res = scaled_vs_unscaled(est, X, y, seed=seed)
        gain = round(res["scaled"] - res["unscaled"], 4)
        table[name] = {
            "unscaled": res["unscaled"],
            "scaled": res["scaled"],
            "gain": gain,
            "needs_scaling": gain > 0.02
        }
    return table


In [2]:
X, y = make_data()
tbl = ablation(X, y, 0)
for name, r in tbl.items():
    print(name, r["unscaled"], r["scaled"], r["gain"], r["needs_scaling"])


knn 0.7222 0.963 0.2408 True
svm 0.6481 1.0 0.3519 True
tree 0.9444 0.9444 0.0 False
forest 1.0 1.0 0.0 False
